# Trino: интерактивный SQL


Выполнять ограниченные запросы, исследовать каталог и читать план без запуска Spark.

Нужны пакет `trino` и доступный сервис Trino с каталогом `lakehouse`. Каждая ячейка создаёт и закрывает собственное соединение; при недоступном сервисе выполнение завершится ошибкой соединения.

Ограничения: Когда нужен быстрый SQL-доступ к нескольким источникам и каталогам. Не возвращайте в Python необрезанный набор и не оставляйте соединение открытым.


## Основные понятия
Trino обращается к каталогам и схемам через SQL. Курсор возвращает строки в процесс Python, поэтому выдача всегда ограничена.


In [ ]:
import trino

## Примеры


**1. Каталог**

Список схем и таблиц помогает обнаружить доступные объекты.



In [ ]:
connection = trino.dbapi.connect(host="trino", port=8080, user="jupyter", catalog="lakehouse", schema="raw")
try:
    cursor = connection.cursor()
    cursor.execute("""
        select table_schema, table_name 
        from lakehouse.information_schema.tables 
        where table_schema in ('raw', 'stage', 'mart') 
        order by table_schema, table_name 
        limit 50
    """)
    print(*cursor.fetchall(), sep="\n")
finally:
    connection.close()

**2. Группировка**

Получите две агрегированные строки по признаку победы.



In [ ]:
connection = trino.dbapi.connect(host="trino", port=8080, user="jupyter", catalog="lakehouse", schema="raw")
try:
    cursor = connection.cursor()
    cursor.execute("""
        select radiant_win, count(*) as matches, round(avg(duration), 1) as average_duration 
        from lakehouse.raw.pg__matches 
        where duration > 0 
        group by radiant_win 
        order by radiant_win
    """)
    print(*cursor.fetchall(), sep="\n")
finally:
    connection.close()

**3. Оконная метрика**

Посчитайте место игроков внутри матча и верните только первые строки.



In [ ]:
import trino

connection = trino.dbapi.connect(host="trino", port=8080, user="jupyter", catalog="lakehouse", schema="raw")
try:
    cursor = connection.cursor()
    cursor.execute("select match_id, hero_id, kills, row_number() over (partition by match_id order by kills desc) as place from lakehouse.raw.pg__match_players where kills is not null limit 30")
    print(*cursor.fetchall(), sep="\n")
finally:
    connection.close()


**4. План**

Запросите план до выполнения тяжёлой операции.



In [ ]:
connection = trino.dbapi.connect(host="trino", port=8080, user="jupyter", catalog="lakehouse", schema="raw")
try:
    cursor = connection.cursor()
    cursor.execute("""
        explain 
        select radiant_win, count(*) 
        from lakehouse.raw.pg__matches 
        where duration > 1800 
        group by radiant_win
    """)
    print("\n".join(row[0] for row in cursor.fetchall()))
finally:
    connection.close()

**5. Описание полей**

Получите сведения о столбцах одной таблицы.



In [ ]:
connection = trino.dbapi.connect(host="trino", port=8080, user="jupyter", catalog="lakehouse", schema="raw")
try:
    cursor = connection.cursor()
    cursor.execute("""
        select column_name, data_type 
        from lakehouse.information_schema.columns
        where table_schema = 'raw' 
            and table_name = 'pg__matches' 
        order by ordinal_position
    """)
    print(*cursor.fetchall(), sep="\n")
finally:
    connection.close()